## Before We Start

Please get a github personal access token. Follow these [instructions](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens#creating-a-personal-access-token-classic)

Copy the key to the clipboard, then add a new line to your .env file:

`GITHUB_PERSONAL_ACCESS_TOKEN=github_pat_`

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import requests

In [2]:
load_dotenv(override=True)

True

## Step 1: Agent Workflow

In [3]:
linter_instructions = """
You are a Code linter. Your task is to analyze the provided code and identify linting issues.
You will also check for styling issues following the Airbnb Style Guide.
"""

security_instructions = """
You are a Security Auditor. Your task is to analyze the provided code and identify security vulnerabilities.
If outputt is provided from 3rd party tools, like SonarQube or Wiz, you will also analyze that output.
You will also check for security issues following the OWASP Top 10.
"""

mentor_insructions = """
You are a Mentor. Your task is to analyze the provided code and provide feedback on how to improve it.
You will also check for best practices and suggest improvements.
Using design patterns and principles.
You will also check for code quality and suggest improvements.
"""

In [6]:
model = 'gpt-4o-mini'

linter_agent = Agent(
    name="linter",
    instructions=linter_instructions,
    model=model
)

security_agent = Agent(
    name="security",
    instructions=security_instructions,
    model=model
)

mentor_agent = Agent(
    name="mentor",
    instructions=mentor_insructions,
    model=model
)


In [10]:
reviewer_insructions = """
You are a PR Reviewer. Your task is to review the provided code and provide feedback.
You will use the linter, security auditor, and mentor agents to analyze the code.
"""

pr_reviewer = Agent(
    name="pr_reviewer",
    instructions=reviewer_insructions,
    model=model,
    tools=[
        function_tool(linter_agent),
        function_tool(security_agent),
        function_tool(mentor_agent)
    ]
)

In [ ]:

@function_tool
def get_pr_contents(pr_number):
    """
    Fetches the contents of a pull request from a hardcoded GitHub repository using a personal access token.

    Args:
        pr_number (int): The pull request number.

    Returns:
        dict: The pull request data as returned by the GitHub API.
    """
    # Hardcoded repository details
    owner = "rado-radoev"
    repo = "agents"
    token = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")

    if not token:
        raise ValueError("GitHub personal access token not found in environment variables.")

    url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}"
    headers = {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.json()